## Keras 3 Built-in Layers for Enterprise Learning and Development

This notebook is organized as a corporate training module for applied ML teams:
- Data scientists building production models
- ML engineers designing reusable architectures
- L&D teams onboarding cross-functional technical staff

Official docs:
- TensorFlow Keras layers catalog: https://www.tensorflow.org/api_docs/python/tf/keras/layers

Enterprise curriculum goals:
1. Understand which Keras layer groups map to specific business problems.
2. Learn stable architecture patterns for tabular, vision, NLP, and sequence systems.
3. Practice safe defaults for performance, robustness, and maintainability.
4. Connect layer choices to enterprise use cases and deployment constraints.

Layer groups covered in this notebook:
- Convolutional layers
- Core layers
- Pooling layers
- Recurrent layers (RNNs)
- Regularization and normalization layers
- Specialized and custom layers
- Preprocessing layers
- Activation layers
- Attention layers
- Reshaping layers
- Merging layers
- Backend-specific layers

Related model-design components included:
- Layer weight initializers
- Layer weight regularizers
- Layer activation functions
- Layer weight constraints

## Theory-Only Reference: Layer Groups and Enterprise Examples

This section is intentionally theory-only and explains purpose, when to use, and practical examples.

### Layer groups covered in this notebook

1. Convolutional layers
Purpose: Learn local spatial patterns through shared kernels.
When to use: Images, document layout maps, spectrogram-like inputs.
Enterprise example: Detect defects in product images on a manufacturing line.

2. Core layers
Purpose: Learn dense feature interactions and projections.
When to use: Tabular modeling and shared heads across tasks.
Enterprise example: Credit risk score from customer profile and transaction features.

3. Pooling layers
Purpose: Reduce spatial or temporal dimension while preserving salient signals.
When to use: Control model size and improve robustness to small shifts.
Enterprise example: Downsample invoice-image features before classification.

4. Recurrent layers (RNNs)
Purpose: Capture order-dependent patterns in sequences.
When to use: Time series, event streams, and ordered text signals.
Enterprise example: Forecast weekly product demand from historical sequences.

5. Regularization and normalization layers
Purpose: Improve generalization and stabilize optimization.
When to use: Overfitting risk, noisy features, or unstable training.
Enterprise example: Dropout plus normalization in a churn model with sparse signals.

6. Specialized and custom layers
Purpose: Embed domain-specific logic not available in standard blocks.
When to use: Business rules, learned gates, or custom transforms.
Enterprise example: Policy-aware gating layer for compliance-sensitive scoring.

7. Preprocessing layers
Purpose: Keep feature transformation inside the model graph.
When to use: Need consistent train-time and serving-time preprocessing.
Enterprise example: Text vectorization and lookup in customer-ticket routing.

8. Activation layers
Purpose: Introduce nonlinearity and control response shape.
When to use: Hidden and output behavior tuning by task type.
Enterprise example: ReLU for hidden layers and sigmoid for default-risk probability.

9. Attention layers
Purpose: Focus on the most relevant parts of a sequence or context.
When to use: Long-context language or multi-step dependency tasks.
Enterprise example: Prioritize key terms in legal-contract clause extraction.

10. Reshaping layers
Purpose: Reformat tensors between architecture blocks.
When to use: Bridge CNN outputs to MLP heads or sequence blocks.
Enterprise example: Flatten pooled visual features before routing head.

11. Merging layers
Purpose: Combine multiple branches or modalities.
When to use: Multi-input models such as tabular plus text or image plus text.
Enterprise example: Fraud model merging card-transaction features with merchant notes.

12. Backend-specific layers
Purpose: Enable backend-coupled integration and serving patterns.
When to use: Deployment requires backend-native model loading or signatures.
Enterprise example: Use TensorFlow-specific serving layer in a platform pipeline.

### Related model-design components included

1. Layer weight initializers
Purpose: Set initial weight distribution for stable gradient flow.
When to use: Always; match initializer to activation depth and architecture style.
Enterprise example: He-style initialization in deep ReLU tabular networks.

2. Layer weight regularizers
Purpose: Penalize overly complex weights to improve generalization.
When to use: Small to medium datasets or high-capacity models.
Enterprise example: L2 regularization in risk models prone to overfit short histories.

3. Layer activation functions
Purpose: Define nonlinear mapping and output semantics.
When to use: Hidden activation by optimization behavior; output activation by target type.
Enterprise example: Softmax for single-label ticket queue assignment.

4. Layer weight constraints
Purpose: Enforce bounded or structured parameter values during training.
When to use: Stability, governance, or safety-sensitive optimization requirements.
Enterprise example: Max-norm constraints in production scoring systems to limit weight explosion.

### Quick decision sequence

1. Choose output activation by target type.
2. Choose hidden activation for optimization stability and signal shape.
3. Add regularization and constraints based on overfitting and governance needs.
4. Select layer groups by data modality and latency budget.

## Guided Theory + Code: Layer Groups with Explanation

This section combines concise theory and commented code in one place.

How to use this section:
1. Read each comment block first (theory and purpose).
2. Run the cell and inspect printed shapes/results.
3. Map each block to a business use case in your domain.

Coverage in the next code cell:
- Convolutional, Core, Pooling, Recurrent, Regularization/Normalization
- Specialized/Custom, Preprocessing, Activation, Attention
- Reshaping, Merging, Backend-specific
- Initializers, Regularizers, Activation functions, Constraints

In [45]:
# Guided walkthrough: each block includes theory comments + practical purpose.

import tensorflow as tf
import numpy as np

np.random.seed(77)
tf.random.set_seed(77)

# Synthetic inputs for tabular, vision, and sequence use cases.
x_tab = tf.random.normal([4, 10])
x_img = tf.random.normal([4, 28, 28, 3])
x_seq = tf.random.normal([4, 12, 16])
x_tok = tf.constant(np.random.randint(0, 200, size=(4, 12)), dtype=tf.int32)
text_in = tf.constant([
    'invoice mismatch for vendor payment',
    'customer requests account unlock',
    'fraud alert for card transaction',
])

# 1) Convolutional layers
# Theory: Convolution learns local spatial patterns with shared filters.
# Purpose: Vision tasks like defect detection and document layout understanding.
conv_feat = tf.keras.layers.Conv2D(12, 3, padding='same', activation='relu')(x_img)

# 2) Core layers
# Theory: Dense mixes all incoming features to model global interactions.
# Purpose: Tabular scoring and projection heads.
core_feat = tf.keras.layers.Dense(16, activation='relu')(x_tab)

# 3) Pooling layers
# Theory: Pooling compresses feature maps and improves shift robustness.
# Purpose: Lower memory/latency in production vision pipelines.
pool_feat = tf.keras.layers.MaxPooling2D(pool_size=2)(conv_feat)

# 4) Recurrent layers (RNNs)
# Theory: RNN family models ordered dependencies over time steps.
# Purpose: Event streams, demand forecasting, sequence state modeling.
rnn_feat = tf.keras.layers.GRU(14)(x_seq)

# 5) Regularization and normalization layers
# Theory: Dropout reduces co-adaptation; normalization stabilizes optimization.
# Purpose: Better generalization and training stability.
reg_norm_block = tf.keras.Sequential([
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
])
reg_norm_feat = reg_norm_block(core_feat, training=True)

# 6) Specialized and custom layers
# Theory: Custom layers encode domain logic when built-ins are insufficient.
# Purpose: Policy-gated transformations for enterprise constraints.
class SimpleGate(tf.keras.layers.Layer):
    def __init__(self, units=16, **kwargs):
        super().__init__(**kwargs)
        self.proj = tf.keras.layers.Dense(units, activation='relu')
        self.gate = tf.keras.layers.Dense(units, activation='sigmoid')

    def call(self, inputs):
        z = self.proj(inputs)
        g = self.gate(inputs)
        return g * z

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'units': self.proj.units})
        return cfg

spec_feat = SimpleGate(16)(x_tab)

# 7) Preprocessing layers
# Theory: In-graph preprocessing keeps train/serve transformations consistent.
# Purpose: Reduce feature drift between training and deployment.
text_vec = tf.keras.layers.TextVectorization(output_mode='int', output_sequence_length=8)
text_vec.adapt(text_in)
prep_feat = text_vec(text_in)

# 8) Activation layers
# Theory: Activation controls nonlinearity and gradient flow behavior.
# Purpose: Fine-tune hidden representation behavior.
act_feat = tf.keras.layers.LeakyReLU(negative_slope=0.1)(tf.keras.layers.Dense(12)(x_tab))

# 9) Attention layers
# Theory: Attention weights context tokens by relevance.
# Purpose: Long-context NLP and document intelligence.
embed_tok = tf.keras.layers.Embedding(input_dim=200, output_dim=18)(x_tok)
attn_feat = tf.keras.layers.MultiHeadAttention(num_heads=3, key_dim=6)(embed_tok, embed_tok)

# 10) Reshaping layers
# Theory: Reshaping bridges architecture interfaces.
# Purpose: Connect CNN outputs to MLP heads.
flat_feat = tf.keras.layers.Flatten()(pool_feat)

# 11) Merging layers
# Theory: Merging combines heterogeneous branches.
# Purpose: Multi-modal models (tabular + vision + text).
branch_tab = tf.keras.layers.Dense(20, activation='relu')(x_tab)
branch_img = tf.keras.layers.Dense(20, activation='relu')(flat_feat)
merge_feat = tf.keras.layers.Concatenate()([branch_tab, branch_img])

# 12) Backend-specific layers
# Theory: Some layers depend on backend-specific runtime features.
# Purpose: Safe compatibility checks before deployment.
backend_has_tfsm = hasattr(tf.keras.layers, 'TFSMLayer')

# Related model-design components
# A) Initializer: controls starting weight distribution.
# B) Regularizer: penalizes complexity to reduce overfit.
# C) Activation function: shapes hidden/output behavior.
# D) Constraint: enforces bounded weights during optimization.
design_layer = tf.keras.layers.Dense(
    10,
    activation=tf.keras.activations.gelu,
    kernel_initializer=tf.keras.initializers.HeNormal(),
    kernel_regularizer=tf.keras.regularizers.l2(1e-4),
    kernel_constraint=tf.keras.constraints.MaxNorm(2.0),
)
design_feat = design_layer(x_tab)

print('Convolutional:', conv_feat.shape)
print('Core:', core_feat.shape)
print('Pooling:', pool_feat.shape)
print('Recurrent:', rnn_feat.shape)
print('Regularization/Normalization:', reg_norm_feat.shape)
print('Specialized custom:', spec_feat.shape)
print('Preprocessing:', prep_feat.shape)
print('Activation:', act_feat.shape)
print('Attention:', attn_feat.shape)
print('Reshaping:', flat_feat.shape)
print('Merging:', merge_feat.shape)
print('Backend-specific TFSMLayer available:', backend_has_tfsm)
print('Design components layer output:', design_feat.shape)
print('Initializer:', type(design_layer.kernel_initializer).__name__)
print('Regularizer:', type(design_layer.kernel_regularizer).__name__)
print('Activation function:', design_layer.activation.__name__)
print('Constraint:', type(design_layer.kernel_constraint).__name__)

Convolutional: (4, 28, 28, 12)
Core: (4, 16)
Pooling: (4, 14, 14, 12)
Recurrent: (4, 14)
Regularization/Normalization: (4, 16)
Specialized custom: (4, 16)
Preprocessing: (3, 8)
Activation: (4, 12)
Attention: (4, 12, 18)
Reshaping: (4, 2352)
Merging: (4, 40)
Backend-specific TFSMLayer available: True
Design components layer output: (4, 10)
Initializer: HeNormal
Regularizer: L2
Activation function: gelu
Constraint: MaxNorm


## Which Layer To Use When (Practical Cheat Sheet)

Use these quick rules in real projects:

1. Tabular classification/regression
- Use: `Dense` stacks as the main backbone.
- Add: `BatchNormalization` or `LayerNormalization` if training is unstable.
- Add: `Dropout` if overfitting appears.
- Output:
  - Binary classification: `Dense(1, activation='sigmoid')`
  - Multi-class single-label: `Dense(num_classes, activation='softmax')`
  - Regression: `Dense(1, activation='linear')`

2. Vision/image tasks
- Use: `Conv2D` to learn local spatial features.
- Add: `MaxPooling2D` for downsampling.
- Add: `GlobalAveragePooling2D` before classifier head for compact representations.
- Use case: defect detection, document image routing.

3. Sequence/time-series tasks
- Use: `LSTM` or `GRU` for ordered temporal dependencies.
- Use `return_sequences=True` for token/time-step labeling.
- Use case: demand forecasting, event log modeling, token tagging.

4. Text/NLP tasks
- Use: `TextVectorization` + `Embedding` as baseline pipeline.
- Add: `MultiHeadAttention` for richer context interactions.
- Use case: intent classification, semantic ranking, clause extraction.

5. Multi-modal tasks (tabular + text/image)
- Build separate branches per modality.
- Merge with `Concatenate` when features are complementary.
- Use case: fraud models combining transaction features and notes.

6. Specialized business logic
- Use a custom `Layer` when rules cannot be represented cleanly with built-ins.
- Implement `get_config()` for portability and serialization.

7. Weight behavior controls
- Initializer: `HeNormal` for ReLU-like networks.
- Regularizer: `l2(1e-4)` as a common safe starting point.
- Constraint: `MaxNorm` when you need bounded weights for stability/governance.

In [46]:
# Decision-demo: minimal code examples for "which layer when"

import tensorflow as tf
import numpy as np

np.random.seed(88)
tf.random.set_seed(88)

# A) Tabular: Dense backbone + output based on target type
x_tab = tf.random.normal([6, 12])
tab_binary = tf.keras.Sequential([
    tf.keras.layers.Dense(24, activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])
print('Tabular binary output shape:', tab_binary(x_tab, training=False).shape)

tab_reg = tf.keras.Sequential([
    tf.keras.layers.Dense(24, activation='relu'),
    tf.keras.layers.Dense(1, activation='linear'),
])
print('Tabular regression output shape:', tab_reg(x_tab, training=False).shape)

# B) Vision: Conv2D + Pooling + GlobalAveragePooling2D
x_img = tf.random.normal([6, 32, 32, 3])
vision_model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(24, 3, padding='same', activation='relu'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(4, activation='softmax'),
])
print('Vision classifier output shape:', vision_model(x_img, training=False).shape)

# C) Sequence/time-series: GRU for ordered dependencies
x_seq = tf.random.normal([6, 15, 10])
seq_model = tf.keras.Sequential([
    tf.keras.layers.GRU(20),
    tf.keras.layers.Dense(3, activation='softmax'),
])
print('Sequence classifier output shape:', seq_model(x_seq, training=False).shape)

# D) Text/NLP: TextVectorization + Embedding + Attention
texts = tf.constant([
    'payment issue in invoice processing',
    'customer asks password reset',
    'urgent fraud alert card blocked',
])
vec = tf.keras.layers.TextVectorization(output_mode='int', output_sequence_length=10)
vec.adapt(texts)
x_tok = vec(texts)
emb = tf.keras.layers.Embedding(input_dim=len(vec.get_vocabulary()), output_dim=16)(x_tok)
attn = tf.keras.layers.MultiHeadAttention(num_heads=2, key_dim=8)(emb, emb)
print('Text attention output shape:', attn.shape)

# E) Multi-modal merge: tabular + image branches with Concatenate
tab_branch = tf.keras.layers.Dense(12, activation='relu')(x_tab)
img_branch = tf.keras.layers.Dense(12, activation='relu')(tf.keras.layers.Flatten()(x_img))
merged = tf.keras.layers.Concatenate()([tab_branch, img_branch])
print('Merged multi-modal feature shape:', merged.shape)

# F) Custom layer for specialized business logic
class RiskGate(tf.keras.layers.Layer):
    def __init__(self, units=12, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.proj = tf.keras.layers.Dense(units, activation='relu')
        self.gate = tf.keras.layers.Dense(units, activation='sigmoid')

    def call(self, inputs):
        return self.gate(inputs) * self.proj(inputs)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'units': self.units})
        return cfg

risk_out = RiskGate(12)(x_tab)
print('Custom layer output shape:', risk_out.shape)

# G) Regularizer + constraint + initializer in one production-style layer
governed = tf.keras.layers.Dense(
    10,
    activation='relu',
    kernel_initializer=tf.keras.initializers.HeNormal(),
    kernel_regularizer=tf.keras.regularizers.l2(1e-4),
    kernel_constraint=tf.keras.constraints.MaxNorm(2.0),
)
print('Governed layer output shape:', governed(x_tab).shape)

Tabular binary output shape: (6, 1)
Tabular regression output shape: (6, 1)
Vision classifier output shape: (6, 4)
Sequence classifier output shape: (6, 3)
Text attention output shape: (3, 10, 16)
Merged multi-modal feature shape: (6, 24)
Custom layer output shape: (6, 12)
Governed layer output shape: (6, 10)


In [47]:
# Common setup
import time
import tensorflow as tf
import numpy as np

tf.random.set_seed(7)
np.random.seed(7)

x_tabular = tf.random.normal([8, 16])
x_image = tf.random.normal([8, 32, 32, 3])
x_tokens = tf.constant(np.random.randint(0, 1000, size=(8, 20)), dtype=tf.int32)

print('TensorFlow:', tf.__version__)
print('x_tabular:', x_tabular.shape, 'x_image:', x_image.shape, 'x_tokens:', x_tokens.shape)

TensorFlow: 2.22.0-dev0+selfbuilt
x_tabular: (8, 16) x_image: (8, 32, 32, 3) x_tokens: (8, 20)


### 1) Purpose and When to Use

Documentation:
- Dense: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
- Conv2D: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D
- Embedding: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding
- LSTM: https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM
- MultiHeadAttention: https://www.tensorflow.org/api_docs/python/tf/keras/layers/MultiHeadAttention

On-point purpose:
- Dense: global feature mixing for tabular or projection heads.
- Conv2D: local spatial pattern learning for images.
- Embedding: integer token id to trainable vector mapping.
- LSTM: sequence modeling with temporal memory.
- MultiHeadAttention: token-to-token contextual interaction.

In [48]:
# Purpose examples
dense = tf.keras.layers.Dense(32, activation='relu')
conv = tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu')
embed = tf.keras.layers.Embedding(input_dim=1000, output_dim=32)
lstm = tf.keras.layers.LSTM(16)
mha = tf.keras.layers.MultiHeadAttention(num_heads=4, key_dim=8)

tab = dense(x_tabular)
img = conv(x_image)
tok = embed(x_tokens)
seq = lstm(tok)
attn = mha(tok, tok)

print('Dense:', tab.shape)
print('Conv2D:', img.shape)
print('Embedding:', tok.shape)
print('LSTM:', seq.shape)
print('MHA:', attn.shape)

Dense: (8, 32)
Conv2D: (8, 32, 32, 16)
Embedding: (8, 20, 32)
LSTM: (8, 16)
MHA: (8, 20, 32)


### 2) Do and Do Not

Common rule:
- Create layer once and reuse.
- Use correct training flag for Dropout and BatchNormalization.

In [49]:
# Do and Do Not examples
shared_dense = tf.keras.layers.Dense(8, activation='relu')
_ = shared_dense(x_tabular)
_ = shared_dense(x_tabular)
print('DO reuse kernel shape:', shared_dense.kernel.shape)

shared_bn = tf.keras.layers.BatchNormalization()
shared_do = tf.keras.layers.Dropout(0.5)
train_out = shared_do(shared_bn(x_tabular, training=True), training=True)
infer_out = shared_do(shared_bn(x_tabular, training=False), training=False)
print('DO train vs infer shapes:', train_out.shape, infer_out.shape)

# DO NOT: recreate layer every step
for i in range(2):
    bad_dense = tf.keras.layers.Dense(8)
    _ = bad_dense(x_tabular)
print('DO NOT pattern shown: new Dense each iteration')

# DO NOT: pass float tokens without validation
bad_tokens = tf.cast(x_tokens, tf.float32)
try:
    tf.debugging.assert_type(bad_tokens, tf.int32)
except Exception as e:
    print('Type validation catches bad token dtype:', type(e).__name__)

DO reuse kernel shape: (16, 8)
DO train vs infer shapes: (8, 16) (8, 16)
DO NOT pattern shown: new Dense each iteration
Type validation catches bad token dtype: TypeError


### 3) Performance Example and Explanation

- tf.function can reduce Python overhead for repeated calls.
- Gains depend on hardware, shapes, and workload size.

In [50]:
# Performance: eager vs graph for repeated Dense calls
perf_x = tf.random.normal([256, 256])
perf_layer = tf.keras.layers.Dense(256, activation='relu')

def eager_forward(x):
    return perf_layer(x)

@tf.function
def graph_forward(x):
    return perf_layer(x)

for _ in range(10):
    _ = eager_forward(perf_x)
    _ = graph_forward(perf_x)

def time_it(fn, x, n=100):
    start = time.perf_counter()
    for _ in range(n):
        _ = fn(x)
    return (time.perf_counter() - start) * 1000.0

eager_ms = time_it(eager_forward, perf_x)
graph_ms = time_it(graph_forward, perf_x)
print(f'Eager 100 calls: {eager_ms:.2f} ms')
print(f'Graph 100 calls: {graph_ms:.2f} ms')
if graph_ms > 0:
    print(f'Speedup eager/graph: {eager_ms / graph_ms:.2f}x')

Eager 100 calls: 47.24 ms
Graph 100 calls: 33.87 ms
Speedup eager/graph: 1.39x


### 4) Normal Cases with Supporting Code

Typical stable defaults for tabular, vision, and NLP pipelines.

In [51]:
# Normal cases
mlp = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1)
])
print('MLP:', mlp(x_tabular, training=True).shape)

vision = tf.keras.Sequential([
    tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(10)
])
print('Vision:', vision(x_image, training=False).shape)

nlp = tf.keras.Sequential([
    tf.keras.layers.Embedding(5000, 64),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(4)
])
tok2 = tf.constant(np.random.randint(0, 5000, size=(8, 30)), dtype=tf.int32)
print('NLP:', nlp(tok2, training=False).shape)

MLP: (8, 1)
Vision: (8, 10)
NLP: (8, 4)


### 5) Edge Cases with Supporting Code

These cases intentionally trigger errors or risky behavior, followed by safe fixes.

In [52]:
# Edge 1: Conv2D wrong rank
conv2d = tf.keras.layers.Conv2D(8, 3)
try:
    _ = conv2d(tf.random.normal([8, 32, 32]))
except Exception as e:
    print('Edge 1 caught:', type(e).__name__)
fixed = tf.expand_dims(tf.random.normal([8, 32, 32]), axis=-1)
print('Edge 1 fix:', conv2d(fixed).shape)

# Edge 2: Embedding out-of-range ids
ids = tf.constant([[0, 10, 101]], dtype=tf.int32)
bad_mask = tf.logical_or(ids < 0, ids >= 100)
print('Edge 2 invalid ids present:', bool(tf.reduce_any(bad_mask).numpy()))
embed2 = tf.keras.layers.Embedding(100, 16)
safe_ids = tf.clip_by_value(ids, 0, 99)
print('Edge 2 fix:', embed2(safe_ids).shape)

# Edge 3: Dropout in inference path
do_layer = tf.keras.layers.Dropout(0.5)
ones = tf.ones([2, 4])
print('Edge 3 train mean:', float(tf.reduce_mean(do_layer(ones, training=True)).numpy()))
print('Edge 3 infer mean:', float(tf.reduce_mean(do_layer(ones, training=False)).numpy()))

# Edge 4: LSTM wrong rank
lstm2 = tf.keras.layers.LSTM(8)
try:
    _ = lstm2(tf.random.normal([8, 16]))
except Exception as e:
    print('Edge 4 caught:', type(e).__name__)
print('Edge 4 fix:', lstm2(tf.random.normal([8, 5, 16])).shape)

Edge 1 caught: ValueError
Edge 1 fix: (8, 30, 30, 8)
Edge 2 invalid ids present: True
Edge 2 fix: (1, 3, 16)
Edge 3 train mean: 0.75
Edge 3 infer mean: 1.0
Edge 4 caught: ValueError
Edge 4 fix: (8, 8)


### 6) Summary

- Pick layers by data type: Dense for tabular, Conv2D for images, Embedding plus sequence layers for tokens.
- Reuse layer instances; avoid recreating inside loops.
- Handle train versus inference behavior explicitly for Dropout and BatchNormalization.
- Benchmark with realistic batch sizes and target hardware.
- Guard edge cases early with shape and dtype checks.

## Machine Learning Concepts with Keras Layers

This section connects Keras layers to core ML concepts and practical NLP tasks.

Coverage:
1. Built-in layers and why they work for specific ML patterns.
2. Custom layers (subclassing) for domain-specific behavior.
3. Use cases: code assistance, text generation, text prediction, NER, POS.
4. Attributes and methods: activation, padding, trainable, weights, get_config, summary-level inspection.
5. Commented code that explains both concept and implementation.

In [53]:
# Shared setup for ML concept examples
import tensorflow as tf
import numpy as np

np.random.seed(13)
tf.random.set_seed(13)

# Tiny corpus with programming and NLP flavored text for quick experimentation
ml_text_corpus = [
    'def add a b return a plus b',
    'def multiply a b return a times b',
    'class user has name and email',
    'function predicts next token from context',
    'entity john is person and london is location',
    'python code completion suggests likely tokens',
    'text generation predicts one token at a time',
    'part of speech tagging labels each token'
]

ml_vectorizer = tf.keras.layers.TextVectorization(
    standardize='lower_and_strip_punctuation',
    split='whitespace',
    output_mode='int',
    output_sequence_length=10,
)
ml_vectorizer.adapt(tf.constant(ml_text_corpus))
ml_vocab = ml_vectorizer.get_vocabulary()
ml_vocab_size = len(ml_vocab)

print('Vocabulary size:', ml_vocab_size)
print('First 12 vocab items:', ml_vocab[:12])

Vocabulary size: 45
First 12 vocab items: ['', '[UNK]', np.str_('a'), np.str_('b'), np.str_('token'), np.str_('return'), np.str_('predicts'), np.str_('is'), np.str_('def'), np.str_('and'), np.str_('user'), np.str_('tokens')]


### 1) Built-in Layers and ML Concept Mapping

Concept to layer mapping:
- Feature interaction: `Dense(activation='relu')`
- Spatial pattern extraction: `Conv2D(padding='same')`
- Sequence representation: `Embedding`
- Context modeling: `LSTM` or `MultiHeadAttention`
- Regularization and stability: `Dropout`, `LayerNormalization`, `BatchNormalization`

Key built-in attributes highlighted here:
- `activation`: introduces non-linearity so models learn complex mappings.
- `padding`: controls output geometry in convolution (`valid` vs `same`).
- `trainable`: controls whether weights are updated during training.

In [54]:
# Built-in layers with commented concept examples

# Dense with activation: good for tabular feature interactions and projection heads
dl_dense = tf.keras.layers.Dense(16, activation='relu')
dl_dense_out = dl_dense(tf.random.normal([4, 8]))

# Conv2D with same padding: preserves spatial size and helps stack deep conv blocks
img_x = tf.random.normal([4, 28, 28, 1])
dl_conv = tf.keras.layers.Conv2D(8, kernel_size=3, padding='same', activation='relu')
dl_conv_out = dl_conv(img_x)

# Embedding + trainable toggle: useful for transfer/freeze workflows
dl_embed = tf.keras.layers.Embedding(input_dim=ml_vocab_size, output_dim=12, trainable=False)
dl_tok = ml_vectorizer(tf.constant(['python code completion suggests token']))
dl_embed_out = dl_embed(dl_tok)

print('Dense output shape:', dl_dense_out.shape)
print('Conv output shape (same padding):', dl_conv_out.shape)
print('Embedding output shape:', dl_embed_out.shape)
print('Embedding trainable:', dl_embed.trainable)

Dense output shape: (4, 16)
Conv output shape (same padding): (4, 28, 28, 8)
Embedding output shape: (1, 10, 12)
Embedding trainable: False


### 2) Customization: Build Your Own Layer

When to customize:
- You need logic not available in built-in layers.
- You need domain-specific transformations with trainable parameters.
- You want reusable behavior with clear config and serialization support.

In [55]:
# Custom layer example: gated residual projection
class GatedResidualDense(tf.keras.layers.Layer):
    def __init__(self, units, activation='relu', **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = tf.keras.activations.get(activation)
        self.proj = tf.keras.layers.Dense(units, activation=activation)
        self.gate = tf.keras.layers.Dense(units, activation='sigmoid')

    def call(self, inputs):
        # Candidate representation from nonlinear projection
        candidate = self.proj(inputs)
        # Gate controls how much of candidate passes through
        gate_value = self.gate(inputs)
        return gate_value * candidate + (1.0 - gate_value) * tf.zeros_like(candidate)

    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units, 'activation': tf.keras.activations.serialize(self.activation)})
        return config

custom_layer = GatedResidualDense(12, activation='relu', name='gated_residual_dense')
custom_out = custom_layer(tf.random.normal([4, 10]))
print('Custom layer output shape:', custom_out.shape)
print('Custom layer config keys:', list(custom_layer.get_config().keys())[:6])

Custom layer output shape: (4, 12)
Custom layer config keys: ['name', 'trainable', 'dtype', 'units', 'activation']


### 3) Use Case: Code Assistance and Text Prediction

ML concept:
- Next-token prediction learns conditional probability $P(w_t \mid w_{<t})$.
- Keras layer stack: `Embedding -> LSTM -> Dense(softmax)`.

In [56]:
# Tiny language model for code assistance / text prediction
ml_token_ids = ml_vectorizer(tf.constant(ml_text_corpus)).numpy()

# Create input/target pairs by shifting sequence by one token
x_lm = ml_token_ids[:, :-1]
y_lm = ml_token_ids[:, 1:]

lm_model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=ml_vocab_size, output_dim=24),
    tf.keras.layers.LSTM(32, return_sequences=True),
    tf.keras.layers.Dense(ml_vocab_size, activation='softmax')
])

lm_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
)

# Small epoch count keeps demo lightweight while showing end-to-end idea
lm_history = lm_model.fit(x_lm, y_lm, epochs=20, verbose=0)
print('Final LM loss:', float(lm_history.history['loss'][-1]))

# Predict next token for a prompt
prompt = 'python code completion'
prompt_ids = ml_vectorizer(tf.constant([prompt])).numpy()[:, :-1]
next_token_probs = lm_model.predict(prompt_ids, verbose=0)
last_pos = int(np.max(np.where(prompt_ids[0] != 0)[0])) if np.any(prompt_ids[0] != 0) else 0

# Avoid special vocabulary slots: 0 is padding, 1 is unknown token
safe_probs = np.array(next_token_probs[0, last_pos], copy=True)
safe_probs[0] = 0.0
safe_probs[1] = 0.0
pred_token_id = int(np.argmax(safe_probs))
print('Prompt:', prompt)
print('Predicted next token:', ml_vocab[pred_token_id])

Final LM loss: 3.703775644302368
Prompt: python code completion
Predicted next token: return


### 4) Use Case: Text Generation

ML concept:
- Autoregressive generation repeatedly predicts one token and appends it to context.
- Same trained next-token model can support generation with greedy decoding.

In [57]:
# Text generation helper using the trained next-token model
ml_id_to_token = ml_vocab


def generate_text(seed_text, steps=5):
    generated = seed_text
    for _ in range(steps):
        # Vectorize current text and run LM
        vec = ml_vectorizer(tf.constant([generated])).numpy()[:, :-1]
        probs = lm_model.predict(vec, verbose=0)
        last_pos = int(np.max(np.where(vec[0] != 0)[0])) if np.any(vec[0] != 0) else 0

        # Avoid generating padding/unknown ids
        safe_probs = np.array(probs[0, last_pos], copy=True)
        safe_probs[0] = 0.0
        safe_probs[1] = 0.0
        next_id = int(np.argmax(safe_probs))

        next_token = ml_id_to_token[next_id]
        generated = generated + ' ' + next_token
    return generated

print('Generated text sample:')
print(generate_text('python code', steps=6))

Generated text sample:
python code is is is is is is


### 5) Use Case: NER and POS Tagging

ML concept:
- Sequence labeling predicts one tag per token.
- Keras layer stack: `Embedding -> BiLSTM(return_sequences=True) -> Dense(tag_classes)`.
- Here we train two heads jointly: NER and POS.

In [58]:
# Minimal commented sample: from text input to token-level predictions

# 1) Prepare tiny sentences for a sequence-labeling style task.
mini_sentences = tf.constant([
    'john works at google',
    'mary lives in paris',
])

# 2) Convert raw text into integer token ids (built-in preprocessing layer).
mini_vec = tf.keras.layers.TextVectorization(
    standardize='lower_and_strip_punctuation',
    split='whitespace',
    output_mode='int',
    output_sequence_length=5,
)
mini_vec.adapt(mini_sentences)
mini_x = mini_vec(mini_sentences)

# 3) Build model with common NLP layers.
# Embedding maps token id -> dense vector.
# BiLSTM contextualizes each token with left+right context.
# Dense over each time-step predicts token-level class probabilities.
mini_model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=len(mini_vec.get_vocabulary()), output_dim=8, mask_zero=True),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(12, return_sequences=True)),
    tf.keras.layers.Dense(4, activation='softmax'),
])

# 4) Dummy token labels for demonstration only (shape: batch, seq_len).
mini_y = tf.constant([
    [1, 0, 2, 3, 0],
    [1, 0, 2, 3, 0],
], dtype=tf.int32)

# 5) Compile and train briefly.
mini_model.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy())
_ = mini_model.fit(mini_x, mini_y, epochs=5, verbose=0)

# 6) Inference: output shape is (batch, seq_len, classes).
mini_probs = mini_model.predict(mini_x, verbose=0)
mini_pred = tf.argmax(mini_probs, axis=-1)

print('Token ids shape:', mini_x.shape)
print('Prediction shape:', mini_probs.shape)
print('Predicted tag ids:\n', mini_pred.numpy())

Token ids shape: (2, 5)
Prediction shape: (2, 5, 4)
Predicted tag ids:
 [[1 2 3 3 1]
 [1 0 2 0 1]]


### Documentation Links + On-Point Explanation

Core documentation:
- Keras layers catalog: https://www.tensorflow.org/api_docs/python/tf/keras/layers
- Dense: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
- Embedding: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding
- LSTM: https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM
- Bidirectional: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Bidirectional
- MultiHeadAttention: https://www.tensorflow.org/api_docs/python/tf/keras/layers/MultiHeadAttention
- TextVectorization: https://www.tensorflow.org/api_docs/python/tf/keras/layers/TextVectorization
- Layer base class (customization): https://www.tensorflow.org/api_docs/python/tf/keras/layers/Layer

On-point explanation:
- Built-in layers: use these first because they are optimized, tested, and composable.
- Custom layer: use when built-ins cannot express your exact transformation.
- Code assistance/text prediction: model learns next-token probability from prior context.
- Text generation: repeatedly apply next-token prediction autoregressively.
- NER/POS: sequence labeling predicts one class per token; `return_sequences=True` is required.
- Attributes: `activation` controls nonlinearity, `padding` controls conv output geometry, `trainable` controls whether weights are updated.

In [59]:
# Toy NER and POS multi-task model
ner_sentences = [
    'john lives in london',
    'mary works at google',
    'alice moved to paris',
    'bob joined microsoft',
]

# NER labels by token: O=0, PER=1, LOC=2, ORG=3
ner_tag_map = {'O': 0, 'PER': 1, 'LOC': 2, 'ORG': 3}
ner_labels_text = [
    ['PER', 'O', 'O', 'LOC'],
    ['PER', 'O', 'O', 'ORG'],
    ['PER', 'O', 'O', 'LOC'],
    ['PER', 'O', 'ORG'],
]

# POS labels by token: NOUN=0, VERB=1, ADP=2, PRON=3
pos_tag_map = {'NOUN': 0, 'VERB': 1, 'ADP': 2, 'PRON': 3}
pos_labels_text = [
    ['NOUN', 'VERB', 'ADP', 'NOUN'],
    ['NOUN', 'VERB', 'ADP', 'NOUN'],
    ['NOUN', 'VERB', 'ADP', 'NOUN'],
    ['NOUN', 'VERB', 'NOUN'],
]

seq_len = 6
seq_vectorizer = tf.keras.layers.TextVectorization(
    standardize='lower_and_strip_punctuation',
    split='whitespace',
    output_mode='int',
    output_sequence_length=seq_len,
)
seq_vectorizer.adapt(tf.constant(ner_sentences))

x_seq = seq_vectorizer(tf.constant(ner_sentences)).numpy()


def encode_tags(tag_sequences, tag_map, max_len):
    arr = np.zeros((len(tag_sequences), max_len), dtype=np.int32)
    for i, tags in enumerate(tag_sequences):
        ids = [tag_map[t] for t in tags]
        arr[i, :min(len(ids), max_len)] = ids[:max_len]
    return arr


y_ner = encode_tags(ner_labels_text, ner_tag_map, seq_len)
y_pos = encode_tags(pos_labels_text, pos_tag_map, seq_len)

# Ignore padding positions during loss computation
mask_weights = (x_seq != 0).astype(np.float32)

num_tokens = len(seq_vectorizer.get_vocabulary())

inputs = tf.keras.Input(shape=(seq_len,), dtype=tf.int32)
x = tf.keras.layers.Embedding(num_tokens, 16, mask_zero=True)(inputs)
x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(24, return_sequences=True))(x)
ner_logits = tf.keras.layers.Dense(len(ner_tag_map), activation='softmax', name='ner')(x)
pos_logits = tf.keras.layers.Dense(len(pos_tag_map), activation='softmax', name='pos')(x)

seq_model = tf.keras.Model(inputs=inputs, outputs=[ner_logits, pos_logits])
seq_model.compile(
    optimizer='adam',
    loss=[
        tf.keras.losses.SparseCategoricalCrossentropy(),
        tf.keras.losses.SparseCategoricalCrossentropy(),
    ],
)

seq_hist = seq_model.fit(
    x_seq,
    [y_ner, y_pos],
    sample_weight=[mask_weights, mask_weights],
    epochs=60,
    verbose=0,
)

for key, values in seq_hist.history.items():
    if key.endswith('loss'):
        print(f'{key}: {float(values[-1]):.4f}')

# Quick inference demonstration
sample = 'john works at google'
sample_ids = seq_vectorizer(tf.constant([sample]))
ner_pred, pos_pred = seq_model.predict(sample_ids, verbose=0)
print('Sample sentence:', sample)
print('Predicted NER tag ids:', np.argmax(ner_pred[0], axis=-1))
print('Predicted POS tag ids:', np.argmax(pos_pred[0], axis=-1))

loss: 2.0909
ner_loss: 1.0257
pos_loss: 1.0653
Sample sentence: john works at google
Predicted NER tag ids: [0 0 0 0 0 0]
Predicted POS tag ids: [0 0 0 0 1 1]


### 6) Layer Attributes and Methods in Practice

Focus attributes:
- `activation`: output non-linearity for representation power.
- `padding`: spatial output size policy in conv layers.
- `trainable`: freeze or unfreeze weights.

Useful methods:
- `get_config()` for serialization metadata.
- `get_weights()` and `set_weights()` for weight inspection or transfer.
- `build(input_shape)` to initialize variables explicitly.
- `summary()` at model level for architecture inspection.

In [60]:
# Attributes and methods demonstration

# activation attribute
attr_dense = tf.keras.layers.Dense(6, activation='tanh', name='attr_dense')
_ = attr_dense(tf.random.normal([2, 4]))
print('Dense activation:', attr_dense.activation.__name__)

# padding attribute
attr_conv = tf.keras.layers.Conv2D(4, kernel_size=3, padding='same', name='attr_conv')
_ = attr_conv(tf.random.normal([2, 16, 16, 1]))
print('Conv2D padding:', attr_conv.padding)

# trainable flag controls whether optimizer updates this layer
attr_embed = tf.keras.layers.Embedding(input_dim=100, output_dim=8, trainable=False, name='attr_embed')
_ = attr_embed(tf.constant([[1, 2, 3]], dtype=tf.int32))
print('Embedding trainable before:', attr_embed.trainable)
attr_embed.trainable = True
print('Embedding trainable after:', attr_embed.trainable)

# method: get_config()
print('Dense config keys sample:', list(attr_dense.get_config().keys())[:7])

# method: get_weights() / set_weights()
w = attr_dense.get_weights()
print('Dense weight tensor count:', len(w))
# Set same weights back just to show transfer API shape safety
attr_dense.set_weights(w)
print('set_weights executed successfully')

# method: build(input_shape) explicitly on a new layer
explicit = tf.keras.layers.Dense(5, activation='relu', name='explicit_build_dense')
explicit.build((None, 7))
print('Explicitly built kernel shape:', explicit.kernel.shape)

# summary() at model level
attr_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(7,)),
    explicit,
    tf.keras.layers.Dense(2, activation='softmax'),
])
attr_model.summary()

Dense activation: tanh
Conv2D padding: same
Embedding trainable before: False
Embedding trainable after: True
Dense config keys sample: ['name', 'trainable', 'dtype', 'units', 'activation', 'use_bias', 'kernel_initializer']
Dense weight tensor count: 2
set_weights executed successfully
Explicitly built kernel shape: (7, 5)


Model: "sequential_33"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ explicit_build_dense (Dense)    │ (None, 5)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_120 (Dense)               │ (None, 2)              │            12 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 52 (208.00 B)

 Trainable params: 52 (208.00 B)

 Non-trainable params: 0 (0.00 B)

### 7) Summary of ML Concepts and Keras Layer Usage

What you now have in this notebook:
1. Built-in layer concepts mapped to ML tasks.
2. A custom layer implementation with `get_config`.
3. Use-case pipelines for code assistance, next-token prediction, and text generation.
4. Sequence labeling examples for NER and POS using multi-output modeling.
5. Practical usage of key attributes (`activation`, `padding`, `trainable`) and methods (`build`, `get_config`, `get_weights`, `set_weights`, `summary`).

Common guidance across use cases:
- Start with built-ins; customize only when behavior is not expressible with composition.
- Keep datasets small for prototyping, then scale model depth and data size.
- Validate shapes, dtypes, and masking behavior early in sequence tasks.
- Treat `trainable` as part of experiment design (freeze vs fine-tune).

## Enterprise Layer Group Catalog: Use Cases and Applied Patterns

This section is designed as a practical corporate training playbook.

### Enterprise capability map

1. Convolutional layers
Use case: visual quality inspection, invoice layout parsing, defect detection.
Key layers: `Conv1D`, `Conv2D`, `SeparableConv2D`, `DepthwiseConv2D`.

2. Core layers
Use case: tabular risk scoring, recommendation scoring heads, shared projection blocks.
Key layers: `Dense`, `Embedding`, `Masking`, `Lambda`.

3. Pooling layers
Use case: dimensionality reduction in vision/audio pipelines to control latency and memory.
Key layers: `MaxPooling1D/2D`, `AveragePooling1D/2D`, `GlobalAveragePooling2D`.

4. Recurrent layers (RNNs)
Use case: demand forecasting, call-center transcript understanding, event stream modeling.
Key layers: `LSTM`, `GRU`, `SimpleRNN`, `Bidirectional`.

5. Regularization and normalization layers
Use case: stabilize training and reduce overfitting in medium-size enterprise datasets.
Key layers: `Dropout`, `GaussianNoise`, `BatchNormalization`, `LayerNormalization`.

6. Specialized and custom layers
Use case: organization-specific business rules, legacy feature transformations, gated logic.
Key layers: subclassed `Layer`, reusable domain blocks.

7. Preprocessing layers
Use case: serving-consistent preprocessing inside the model graph.
Key layers: `TextVectorization`, `Normalization`, `Discretization`, `StringLookup`.

8. Activation layers
Use case: control non-linearity behavior and gradient properties.
Key layers: `ReLU`, `LeakyReLU`, `PReLU`, `Softmax`.

9. Attention layers
Use case: document intelligence, semantic search ranking, long-context modeling.
Key layers: `Attention`, `AdditiveAttention`, `MultiHeadAttention`.

10. Reshaping layers
Use case: bridge representation formats across CNN/RNN/MLP blocks.
Key layers: `Flatten`, `Reshape`, `Permute`, `RepeatVector`.

11. Merging layers
Use case: multi-modal or multi-branch architectures in enterprise AI systems.
Key layers: `Add`, `Concatenate`, `Average`, `Multiply`.

12. Backend-specific layers
Use case: backend-coupled deployment behavior.
Example: `TFSMLayer` (TensorFlow backend) for loading SavedModel signatures.

### Layer design controls

- Layer weight initializers: `HeNormal`, `GlorotUniform`, `Orthogonal`
- Layer weight regularizers: `l1`, `l2`, `l1_l2`
- Layer activation functions: `relu`, `gelu`, `tanh`, `softmax`
- Layer weight constraints: `MaxNorm`, `NonNeg`, `UnitNorm`

The next code cell shows compact, meaningful examples for each group.

In [61]:
# Enterprise layer-group examples with practical use-case framing

import tensorflow as tf
import numpy as np

np.random.seed(21)
tf.random.set_seed(21)

# Synthetic enterprise-style inputs
x_tab = tf.random.normal([6, 12])
x_img = tf.random.normal([6, 32, 32, 3])
x_seq = tf.random.normal([6, 10, 16])
x_tok = tf.constant(np.random.randint(0, 300, size=(6, 10)), dtype=tf.int32)
text_batch = tf.constant([
    'invoice total mismatch for vendor',
    'urgent support ticket for payment gateway',
    'contract clause review for legal team',
])

# 1) Convolutional layers: visual inspection/classification
conv_block = tf.keras.Sequential([
    tf.keras.layers.Conv2D(16, 3, padding='same', activation='relu'),
    tf.keras.layers.SeparableConv2D(16, 3, padding='same', activation='relu'),
])
conv_out = conv_block(x_img)

# 2) Core layers: tabular scoring head
core_out = tf.keras.layers.Dense(20, activation='relu')(x_tab)

# 3) Pooling layers: downsample for compute efficiency
pool_out = tf.keras.layers.MaxPooling2D(pool_size=2)(conv_out)

# 4) Recurrent layers: sequence understanding for event streams
rnn_out = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(12))(x_seq)

# 5) Regularization and normalization layers: training stability
reg_norm = tf.keras.Sequential([
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.LayerNormalization(),
])
reg_norm_out = reg_norm(core_out, training=True)

# 6) Specialized and custom layers: policy-gated business signal
class PolicyGate(tf.keras.layers.Layer):
    def __init__(self, units=12, **kwargs):
        super().__init__(**kwargs)
        self.proj = tf.keras.layers.Dense(units, activation='relu')
        self.gate = tf.keras.layers.Dense(units, activation='sigmoid')

    def call(self, inputs):
        z = self.proj(inputs)
        g = self.gate(inputs)
        return g * z

special_out = PolicyGate(12)(x_tab)

# 7) Preprocessing layers: in-graph text preprocessing
text_vec = tf.keras.layers.TextVectorization(output_mode='int', output_sequence_length=8)
text_vec.adapt(text_batch)
prep_out = text_vec(text_batch)

# 8) Activation layers: explicit activation choice
act_out = tf.keras.layers.PReLU()(tf.keras.layers.Dense(12)(x_tab))

# 9) Attention layers: contextual token interaction
tok_embed = tf.keras.layers.Embedding(input_dim=300, output_dim=24)(x_tok)
attn_out = tf.keras.layers.MultiHeadAttention(num_heads=4, key_dim=6)(tok_embed, tok_embed)

# 10) Reshaping layers: bridge CNN and MLP interfaces
reshape_out = tf.keras.layers.Flatten()(pool_out)

# 11) Merging layers: combine branches in multi-modal systems
tab_proj = tf.keras.layers.Dense(32, activation='relu')(x_tab)
img_proj = tf.keras.layers.Dense(32, activation='relu')(reshape_out)
merge_out = tf.keras.layers.Concatenate()([tab_proj, img_proj])

# 12) Backend-specific layers: TensorFlow-specific availability check
has_tfsm_layer = hasattr(tf.keras.layers, 'TFSMLayer')

# Layer design controls: initializers, regularizers, activations, constraints
governed_dense = tf.keras.layers.Dense(
    16,
    activation=tf.keras.activations.gelu,
    kernel_initializer=tf.keras.initializers.HeNormal(),
    kernel_regularizer=tf.keras.regularizers.l2(1e-4),
    kernel_constraint=tf.keras.constraints.MaxNorm(max_value=2.0),
)
gov_out = governed_dense(x_tab)

print('Convolutional layers output:', conv_out.shape)
print('Core layers output:', core_out.shape)
print('Pooling layers output:', pool_out.shape)
print('Recurrent layers output:', rnn_out.shape)
print('Regularization/Normalization output:', reg_norm_out.shape)
print('Specialized custom layer output:', special_out.shape)
print('Preprocessing layer output:', prep_out.shape)
print('Activation layers output:', act_out.shape)
print('Attention layers output:', attn_out.shape)
print('Reshaping layers output:', reshape_out.shape)
print('Merging layers output:', merge_out.shape)
print('Backend-specific TFSMLayer available:', has_tfsm_layer)
print('Governed dense output:', gov_out.shape)

print('Initializer used:', type(governed_dense.kernel_initializer).__name__)
print('Regularizer used:', type(governed_dense.kernel_regularizer).__name__)
print('Activation used:', governed_dense.activation.__name__)
print('Constraint used:', type(governed_dense.kernel_constraint).__name__)

Convolutional layers output: (6, 32, 32, 16)
Core layers output: (6, 20)
Pooling layers output: (6, 16, 16, 16)
Recurrent layers output: (6, 24)
Regularization/Normalization output: (6, 20)
Specialized custom layer output: (6, 12)
Preprocessing layer output: (3, 8)
Activation layers output: (6, 12)
Attention layers output: (6, 10, 24)
Reshaping layers output: (6, 4096)
Merging layers output: (6, 64)
Backend-specific TFSMLayer available: True
Governed dense output: (6, 16)
Initializer used: HeNormal
Regularizer used: L2
Activation used: gelu
Constraint used: MaxNorm


## Corporate Training Track: Beginner to Advanced

This training path is optimized for enterprise onboarding and capability development.

### Level 1: Beginner (Foundations)
Learning objectives:
- Identify when to use core, convolutional, pooling, and recurrent layers.
- Understand shape flow and layer outputs.
- Use safe defaults for activations and normalization.

Hands-on checkpoint:
1. Build a 2-layer tabular model with `Dense` + `Dropout`.
2. Build a 2-block vision model with `Conv2D` + `MaxPooling2D`.
3. Explain output shapes for both models.

Enterprise readiness criteria:
- Can justify layer choice based on data modality.
- Can detect obvious shape mismatch before training.

### Level 2: Intermediate (Applied System Design)
Learning objectives:
- Combine preprocessing, sequence modeling, and attention.
- Apply regularizers, initializers, and constraints intentionally.
- Build multi-branch models using merging layers.

Hands-on checkpoint:
1. Add `TextVectorization` and `Embedding` to an NLP pipeline.
2. Add `MultiHeadAttention` to improve contextual modeling.
3. Merge tabular and text branches with `Concatenate`.

Enterprise readiness criteria:
- Can design a small multi-modal architecture.
- Can explain trade-offs between quality and inference cost.

### Level 3: Advanced (Production and Governance)
Learning objectives:
- Implement custom layers for domain logic.
- Add constraints and regularization for model governance.
- Define deployment checks for backend-specific compatibility.

Hands-on checkpoint:
1. Implement a custom layer with `get_config`.
2. Add `kernel_regularizer` and `kernel_constraint` to critical layers.
3. Validate backend-specific capabilities before export.

Enterprise readiness criteria:
- Can produce maintainable model code with clear extension points.
- Can define minimum quality gates for release.

## Enterprise Case Studies (Independent Code Flow + Reuse Guidance)

This section now enforces independent code flow per case study while documenting reusable parts.

### Independence rule
- Each case study has its own:
  - data generation and preprocessing
  - train/test split
  - model builder
  - training loop
  - evaluation
  - model save path
  - early stopping and checkpoint callbacks

### Reuse rule (allowed and recommended)
Reusable utilities can be shared across case studies to reduce duplication and enforce enterprise consistency:
1. Distributed strategy selector
2. Dataset split helper
3. Callback policy builder (EarlyStopping + ModelCheckpoint)
4. Hyperparameter config structure

### Callback policy standard
- Early stopping:
  - monitor: validation loss
  - restore best weights: true
  - patience: small (for demo), larger in production
- Checkpoint policy:
  - save best only
  - monitor: validation loss
  - one artifact per case study

### Case Study 1: Financial Risk Triage (Tabular + Text)
- Topic: risk prioritization for support operations.
- Keywords: tabular, text, multimodal, binary classification.
- Concepts involved: branch fusion, regularization, constraints, probability outputs.
- Purpose and output: predict high-risk probability.
- Features included: tabular customer metrics + ticket text indicators.
- Distributed training/testing: MirroredStrategy, explicit train/test datasets.

### Case Study 2: Document Routing (Vision + Text)
- Topic: automated enterprise document routing.
- Keywords: vision, text, multiclass routing, fusion architecture.
- Concepts involved: Conv2D feature extraction, embedding, branch fusion.
- Purpose and output: route to one business department class.
- Features included: image-like document tensor + text summary signal.
- Distributed training/testing: MirroredStrategy, explicit train/test datasets.

The next code cell implements both case studies independently and prints documentation outputs with hyperparameters, callback policy, and saved model artifacts.

In [62]:
# Enterprise case studies: independent flows with reusable utilities.

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import tensorflow as tf

np.random.seed(121)
tf.random.set_seed(121)


@dataclass(frozen=True)
class CaseConfig:
    """Shared hyperparameter schema; values can differ per case study."""

    epochs: int
    batch_size: int
    learning_rate: float
    dropout_rate: float
    l2_weight: float
    train_split: float
    early_stopping_patience: int


CASE1_CFG = CaseConfig(epochs=8, batch_size=16, learning_rate=1e-3, dropout_rate=0.2, l2_weight=1e-4, train_split=0.8, early_stopping_patience=2)
CASE2_CFG = CaseConfig(epochs=8, batch_size=16, learning_rate=1e-3, dropout_rate=0.2, l2_weight=1e-4, train_split=0.8, early_stopping_patience=2)


CLASS_NAMES_CASE2 = ['finance', 'legal', 'it_onboarding', 'customer_support']


def select_strategy() -> tf.distribute.Strategy:
    """Reusable utility: distributed strategy selector."""
    try:
        return tf.distribute.MirroredStrategy()
    except Exception:
        return tf.distribute.OneDeviceStrategy('/cpu:0')


def split_indices(n: int, train_split: float) -> Tuple[np.ndarray, np.ndarray]:
    """Reusable utility: deterministic train/test split helper."""
    idx = np.arange(n)
    np.random.shuffle(idx)
    cut = int(n * train_split)
    return idx[:cut], idx[cut:]


def make_callbacks(checkpoint_path: Path, patience: int):
    """Reusable utility: enterprise callback policy (early stopping + checkpoint)."""
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=patience,
        restore_best_weights=True,
        verbose=0,
    )
    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=True,
        verbose=0,
    )
    return [early_stopping, checkpoint]


def binary_confusion_report(y_true: tf.Tensor, y_prob: tf.Tensor, threshold: float = 0.5) -> Dict[str, float]:
    """Compute binary confusion metrics without external dependencies."""
    y_true_np = np.array(y_true).astype(np.int32)
    y_pred_np = (np.array(y_prob).reshape(-1) >= threshold).astype(np.int32)

    tn = int(np.sum((y_true_np == 0) & (y_pred_np == 0)))
    fp = int(np.sum((y_true_np == 0) & (y_pred_np == 1)))
    fn = int(np.sum((y_true_np == 1) & (y_pred_np == 0)))
    tp = int(np.sum((y_true_np == 1) & (y_pred_np == 1)))

    eps = 1e-8
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2.0 * precision * recall / (precision + recall + eps)

    return {
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'tp': tp,
        'precision_pos': float(precision),
        'recall_pos': float(recall),
        'f1_pos': float(f1),
    }


def multiclass_confusion_report(y_true: tf.Tensor, y_prob: tf.Tensor, class_names) -> Tuple[np.ndarray, str]:
    """Compute confusion matrix and per-class precision/recall/F1 report."""
    y_true_np = np.array(y_true).astype(np.int32)
    y_pred_np = np.argmax(np.array(y_prob), axis=1).astype(np.int32)
    num_classes = len(class_names)

    cm = tf.math.confusion_matrix(y_true_np, y_pred_np, num_classes=num_classes).numpy()

    lines = []
    lines.append('class | precision | recall | f1 | support')
    lines.append('----- | --------- | ------ | -- | -------')

    eps = 1e-8
    for i, name in enumerate(class_names):
        tp = float(cm[i, i])
        fp = float(np.sum(cm[:, i]) - tp)
        fn = float(np.sum(cm[i, :]) - tp)
        support = int(np.sum(cm[i, :]))

        precision = tp / (tp + fp + eps)
        recall = tp / (tp + fn + eps)
        f1 = 2.0 * precision * recall / (precision + recall + eps)

        lines.append(f'{name} | {precision:.3f} | {recall:.3f} | {f1:.3f} | {support}')

    return cm, '\n'.join(lines)


def print_feature_summary(case_name: str, train_x: Dict[str, tf.Tensor], test_x: Dict[str, tf.Tensor]) -> None:
    """Print the actual features each case study provides when executed."""
    print(f'Features provided by {case_name}:')
    for feature_name, tensor in train_x.items():
        arr = np.array(tensor)
        sample_value = arr[0]
        sample_text = str(sample_value)
        if len(sample_text) > 120:
            sample_text = sample_text[:117] + '...'
        print(f'- {feature_name}: train_shape={arr.shape}, test_shape={np.array(test_x[feature_name]).shape}, dtype={arr.dtype}')
        print(f'  sample[{feature_name}]: {sample_text}')


ARTIFACT_DIR = Path('artifacts') / 'case_studies'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================
# Case Study 1 (Independent flow): Financial Risk Triage
# =============================================================


def case1_build_data(n: int = 120):
    """Case 1 data pipeline: tabular + text -> binary label."""
    x_tab = tf.random.normal([n, 8])
    x_text = tf.constant([
        'urgent refund payment failed' if i % 3 == 0 else
        'password reset account unlock' if i % 3 == 1 else
        'suspicious transaction reported'
        for i in range(n)
    ])
    score = 0.9 * x_tab[:, 0] - 0.6 * x_tab[:, 1] + 0.4 * x_tab[:, 2]
    text_signal = tf.cast(tf.equal(tf.math.mod(tf.range(n), 3), 2), tf.float32)
    y = tf.cast(score + text_signal > 0.2, tf.float32)
    return x_tab, x_text, y


def case1_build_model(vocab_size: int, cfg: CaseConfig):
    """Case 1 model: branch fusion with governed binary head."""
    inp_tab = tf.keras.Input(shape=(8,), name='tab')
    inp_text = tf.keras.Input(shape=(12,), dtype=tf.int64, name='text_ids')

    tab_branch = tf.keras.layers.Dense(
        24,
        activation='relu',
        kernel_initializer='he_normal',
        kernel_regularizer=tf.keras.regularizers.l2(cfg.l2_weight),
    )(inp_tab)
    tab_branch = tf.keras.layers.LayerNormalization()(tab_branch)
    tab_branch = tf.keras.layers.Dropout(cfg.dropout_rate)(tab_branch)

    text_branch = tf.keras.layers.Embedding(vocab_size, 16)(inp_text)
    text_branch = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(10))(text_branch)

    merged = tf.keras.layers.Concatenate()([tab_branch, text_branch])
    hidden = tf.keras.layers.Dense(16, activation='relu', kernel_constraint=tf.keras.constraints.MaxNorm(2.0))(merged)
    out = tf.keras.layers.Dense(1, activation='sigmoid')(hidden)

    model = tf.keras.Model(inputs={'tab': inp_tab, 'text_ids': inp_text}, outputs=out, name='case1_risk_triage')
    model.compile(optimizer=tf.keras.optimizers.Adam(cfg.learning_rate), loss='binary_crossentropy', metrics=['accuracy'])
    return model


# Independent training/testing flow for Case 1
case1_strategy = select_strategy()
case1_x_tab, case1_x_text, case1_y = case1_build_data()

case1_vectorizer = tf.keras.layers.TextVectorization(output_mode='int', output_sequence_length=12)
case1_vectorizer.adapt(case1_x_text)
case1_text_ids = case1_vectorizer(case1_x_text)

case1_train_idx, case1_test_idx = split_indices(len(case1_y), CASE1_CFG.train_split)
case1_train_x = {
    'tab': tf.gather(case1_x_tab, case1_train_idx),
    'text_ids': tf.gather(case1_text_ids, case1_train_idx),
}
case1_test_x = {
    'tab': tf.gather(case1_x_tab, case1_test_idx),
    'text_ids': tf.gather(case1_text_ids, case1_test_idx),
}
case1_train_y = tf.gather(case1_y, case1_train_idx)
case1_test_y = tf.gather(case1_y, case1_test_idx)

with case1_strategy.scope():
    case1_model = case1_build_model(vocab_size=len(case1_vectorizer.get_vocabulary()), cfg=CASE1_CFG)

case1_ckpt = ARTIFACT_DIR / 'case1_best.weights.h5'
case1_callbacks = make_callbacks(case1_ckpt, CASE1_CFG.early_stopping_patience)

case1_hist = case1_model.fit(
    case1_train_x,
    case1_train_y,
    validation_data=(case1_test_x, case1_test_y),
    epochs=CASE1_CFG.epochs,
    batch_size=CASE1_CFG.batch_size,
    callbacks=case1_callbacks,
    verbose=0,
)
case1_test_loss, case1_test_acc = case1_model.evaluate(case1_test_x, case1_test_y, verbose=0)

case1_saved_model_path = ARTIFACT_DIR / 'case1_model.keras'
case1_model.save(case1_saved_model_path)

case1_test_prob = case1_model.predict(case1_test_x, verbose=0)
case1_report = binary_confusion_report(case1_test_y, case1_test_prob)

print('\n=== Case Study 1 Documentation ===')
print('Topic: Financial Risk Triage')
print('Keywords: tabular, text, multimodal, binary')
print('Concepts: fusion, regularization, constraints, probability output')
print('Purpose: predict high-risk probability per ticket')
print('Distributed strategy:', type(case1_strategy).__name__)
print('Train/Test split:', f"{CASE1_CFG.train_split:.0%}/{(1.0 - CASE1_CFG.train_split):.0%}")
print('Hyperparameters:', CASE1_CFG)
print('Callback policy: EarlyStopping(val_loss, restore_best_weights=True) + best-only checkpoint')
print('Output: probability in [0,1]')
print_feature_summary('Case Study 1', case1_train_x, case1_test_x)
print('Final train loss:', float(case1_hist.history['loss'][-1]))
print('Final val loss:', float(case1_hist.history['val_loss'][-1]))
print('Test accuracy:', float(case1_test_acc))
print('Binary confusion metrics:', case1_report)
print('Saved model:', str(case1_saved_model_path))
print('Saved checkpoint:', str(case1_ckpt))
print('DO: keep output semantics explicit and validate on test split')
print('DO NOT: skip held-out test evaluation')


# =============================================================
# Case Study 2 (Independent flow): Document Routing
# =============================================================


def case2_build_data(n: int = 120):
    """Case 2 data pipeline: image-like tensor + text -> multiclass label."""
    x_img = tf.random.normal([n, 28, 28, 1])
    x_text = tf.constant([
        'invoice payment review finance' if i % 4 == 0 else
        'legal contract clause request' if i % 4 == 1 else
        'employee onboarding access' if i % 4 == 2 else
        'customer escalation support'
        for i in range(n)
    ])
    y = tf.cast(tf.math.mod(tf.range(n), 4), tf.int32)
    return x_img, x_text, y


def case2_build_model(vocab_size: int, cfg: CaseConfig):
    """Case 2 model: vision branch + text branch with multiclass head."""
    inp_img = tf.keras.Input(shape=(28, 28, 1), name='img')
    inp_text = tf.keras.Input(shape=(12,), dtype=tf.int64, name='text_ids')

    img_branch = tf.keras.layers.Conv2D(12, 3, padding='same', activation='relu')(inp_img)
    img_branch = tf.keras.layers.MaxPooling2D(2)(img_branch)
    img_branch = tf.keras.layers.Conv2D(20, 3, padding='same', activation='relu')(img_branch)
    img_branch = tf.keras.layers.GlobalAveragePooling2D()(img_branch)

    text_branch = tf.keras.layers.Embedding(vocab_size, 16)(inp_text)
    text_branch = tf.keras.layers.GRU(10)(text_branch)

    merged = tf.keras.layers.Concatenate()([img_branch, text_branch])
    merged = tf.keras.layers.Dropout(cfg.dropout_rate)(merged)
    out = tf.keras.layers.Dense(4, activation='softmax')(merged)

    model = tf.keras.Model(inputs={'img': inp_img, 'text_ids': inp_text}, outputs=out, name='case2_doc_router')
    model.compile(optimizer=tf.keras.optimizers.Adam(cfg.learning_rate), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model


# Independent training/testing flow for Case 2
case2_strategy = select_strategy()
case2_x_img, case2_x_text, case2_y = case2_build_data()

case2_vectorizer = tf.keras.layers.TextVectorization(output_mode='int', output_sequence_length=12)
case2_vectorizer.adapt(case2_x_text)
case2_text_ids = case2_vectorizer(case2_x_text)

case2_train_idx, case2_test_idx = split_indices(len(case2_y), CASE2_CFG.train_split)
case2_train_x = {
    'img': tf.gather(case2_x_img, case2_train_idx),
    'text_ids': tf.gather(case2_text_ids, case2_train_idx),
}
case2_test_x = {
    'img': tf.gather(case2_x_img, case2_test_idx),
    'text_ids': tf.gather(case2_text_ids, case2_test_idx),
}
case2_train_y = tf.gather(case2_y, case2_train_idx)
case2_test_y = tf.gather(case2_y, case2_test_idx)

with case2_strategy.scope():
    case2_model = case2_build_model(vocab_size=len(case2_vectorizer.get_vocabulary()), cfg=CASE2_CFG)

case2_ckpt = ARTIFACT_DIR / 'case2_best.weights.h5'
case2_callbacks = make_callbacks(case2_ckpt, CASE2_CFG.early_stopping_patience)

case2_hist = case2_model.fit(
    case2_train_x,
    case2_train_y,
    validation_data=(case2_test_x, case2_test_y),
    epochs=CASE2_CFG.epochs,
    batch_size=CASE2_CFG.batch_size,
    callbacks=case2_callbacks,
    verbose=0,
)
case2_test_loss, case2_test_acc = case2_model.evaluate(case2_test_x, case2_test_y, verbose=0)

case2_saved_model_path = ARTIFACT_DIR / 'case2_model.keras'
case2_model.save(case2_saved_model_path)

case2_test_prob = case2_model.predict(case2_test_x, verbose=0)
case2_cm, case2_report = multiclass_confusion_report(case2_test_y, case2_test_prob, CLASS_NAMES_CASE2)

print('\n=== Case Study 2 Documentation ===')
print('Topic: Document Routing')
print('Keywords: vision, text, multiclass, routing')
print('Concepts: Conv2D extraction, embedding, branch fusion, softmax output')
print('Purpose: route document to one of 4 departments')
print('Distributed strategy:', type(case2_strategy).__name__)
print('Train/Test split:', f"{CASE2_CFG.train_split:.0%}/{(1.0 - CASE2_CFG.train_split):.0%}")
print('Hyperparameters:', CASE2_CFG)
print('Callback policy: EarlyStopping(val_loss, restore_best_weights=True) + best-only checkpoint')
print('Output: 4-class probability distribution')
print_feature_summary('Case Study 2', case2_train_x, case2_test_x)
print('Final train loss:', float(case2_hist.history['loss'][-1]))
print('Final val loss:', float(case2_hist.history['val_loss'][-1]))
print('Test accuracy:', float(case2_test_acc))
print('Confusion matrix:\n', case2_cm)
print('Per-class report:\n' + case2_report)
print('Saved model:', str(case2_saved_model_path))
print('Saved checkpoint:', str(case2_ckpt))
print('DO: keep preprocessing and class semantics explicit')
print('DO NOT: ignore class-wise behavior in evaluation')


print('\nReusable code across independent flows:')
print('1) select_strategy() for distributed setup')
print('2) split_indices() for train/test split policy')
print('3) make_callbacks() for early stopping + checkpoint policy')
print('4) CaseConfig dataclass for hyperparameter governance')
print('5) confusion report helpers for consistent testing diagnostics')

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0',)

=== Case Study 1 Documentation ===
Topic: Financial Risk Triage
Keywords: tabular, text, multimodal, binary
Concepts: fusion, regularization, constraints, probability output
Purpose: predict high-risk probability per ticket
Distributed strategy: MirroredStrategy
Train/Test split: 80%/20%
Hyperparameters: CaseConfig(epochs=8, batch_size=16, learning_rate=0.001, dropout_rate=0.2, l2_weight=0.0001, train_split=0.8, early_stopping_patience=2)
Callback policy: EarlyStopping(val_loss, restore_best_weights=True) + best-only checkpoint
Output: probability in [0,1]
Features provided by Case Study 1:
- tab: train_shape=(96, 8), test_shape=(24, 8), dtype=float32
  sample[tab]: [ 0.14089847 -1.5133891   1.5089595   0.49494767 -1.8681012   0.03790881
 -1.8682888   0.56832117]
- text_ids: train_shape=(96, 12), test_shape=(24, 12), dtype=int64
  sample[text_ids]: [10  6 12  3  0  0  0  0  0  0  0  0

## Running Saved Models: Feature Inputs and Inference Output

This section shows how to run the saved models and what features must be passed.

Case Study 1 model input features:
- `tab`: numeric tabular tensor with shape `(batch, 8)`
- `text_ids`: token-id tensor with shape `(batch, 12)`
Output:
- binary risk probability with shape `(batch, 1)`

Case Study 2 model input features:
- `img`: image tensor with shape `(batch, 28, 28, 1)`
- `text_ids`: token-id tensor with shape `(batch, 12)`
Output:
- multiclass probabilities with shape `(batch, 4)`

The next code cell loads each saved model from disk and runs inference using test-feature samples.

In [63]:
# Load saved models and run inference with explicit feature dictionaries.

from pathlib import Path
import numpy as np
import tensorflow as tf

case1_model_path = Path('artifacts/case_studies/case1_model.keras')
case2_model_path = Path('artifacts/case_studies/case2_model.keras')

if not case1_model_path.exists() or not case2_model_path.exists():
    raise FileNotFoundError(
        'Saved model artifacts not found. Run the case-study training cell first to create .keras files.'
    )

# If test features are not present in memory, stop with a clear message.
required_vars = ['case1_test_x', 'case2_test_x']
missing = [name for name in required_vars if name not in globals()]
if missing:
    raise RuntimeError(
        f'Missing in-memory feature sets: {missing}. Run the case-study cell first to generate test features.'
    )

loaded_case1 = tf.keras.models.load_model(case1_model_path)
loaded_case2 = tf.keras.models.load_model(case2_model_path)

# Use small batches for quick inference demo.
case1_features = {
    'tab': case1_test_x['tab'][:3],
    'text_ids': case1_test_x['text_ids'][:3],
}
case2_features = {
    'img': case2_test_x['img'][:3],
    'text_ids': case2_test_x['text_ids'][:3],
}

case1_pred = loaded_case1.predict(case1_features, verbose=0)
case2_pred = loaded_case2.predict(case2_features, verbose=0)

print('=== Saved Model Inference: Case Study 1 ===')
print('Required feature keys:', list(case1_features.keys()))
print('tab shape:', tuple(case1_features['tab'].shape), 'text_ids shape:', tuple(case1_features['text_ids'].shape))
print('Output shape:', case1_pred.shape)
print('Predicted risk probabilities (first 3):', np.round(case1_pred.reshape(-1), 4))

print('\n=== Saved Model Inference: Case Study 2 ===')
print('Required feature keys:', list(case2_features.keys()))
print('img shape:', tuple(case2_features['img'].shape), 'text_ids shape:', tuple(case2_features['text_ids'].shape))
print('Output shape:', case2_pred.shape)
print('Predicted class probabilities (first sample):', np.round(case2_pred[0], 4))
print('Predicted class ids (first 3):', np.argmax(case2_pred, axis=1))

print('\nFeature explanation while running model:')
print('- Case 1 uses tabular risk factors (`tab`) and encoded ticket tokens (`text_ids`) to output risk probability.')
print('- Case 2 uses image signal (`img`) and encoded text (`text_ids`) to output routing class probabilities.')

=== Saved Model Inference: Case Study 1 ===
Required feature keys: ['tab', 'text_ids']
tab shape: (3, 8) text_ids shape: (3, 12)
Output shape: (3, 1)
Predicted risk probabilities (first 3): [0.7822 0.641  0.8984]

=== Saved Model Inference: Case Study 2 ===
Required feature keys: ['img', 'text_ids']
img shape: (3, 28, 28, 1) text_ids shape: (3, 12)
Output shape: (3, 4)
Predicted class probabilities (first sample): [0.262  0.2273 0.2252 0.2855]
Predicted class ids (first 3): [3 3 3]

Feature explanation while running model:
- Case 1 uses tabular risk factors (`tab`) and encoded ticket tokens (`text_ids`) to output risk probability.
- Case 2 uses image signal (`img`) and encoded text (`text_ids`) to output routing class probabilities.


## Assessment: Knowledge Checks with Answer Key

### Section A: Concept checks
1. Why would you choose `LayerNormalization` over `BatchNormalization` in some sequence models?
2. What is the practical impact of `padding='same'` in convolutional layers?
3. Why is `return_sequences=True` required for token-level tasks such as NER?
4. What does `kernel_constraint=MaxNorm(...)` help control during training?

### Section B: Architecture decisions
1. For a multi-modal risk model, when should you use `Concatenate` vs `Add`?
2. Which initializer is often preferred for deep ReLU stacks and why?
3. Where would you apply regularization first in an enterprise model with overfitting?

### Section C: Implementation checks
1. Add one custom layer that encapsulates business logic and supports `get_config`.
2. Demonstrate one preprocessing layer that can run consistently at inference time.
3. Show one backend-specific compatibility check before deployment.

### Suggested answer key
- A1: `LayerNormalization` is independent of batch statistics and can be more stable for variable or small batch sequence workloads.
- A2: `same` keeps spatial dimensions approximately unchanged, simplifying deep stack design.
- A3: Token-level heads require one output per time step, so full sequence outputs are needed.
- A4: MaxNorm limits weight magnitude to improve training stability and reduce extreme parameter growth.
- B1: `Concatenate` preserves complementary features; `Add` is suitable when branches are aligned and semantically similar.
- B2: `HeNormal` or related He initializers are strong defaults for ReLU-like activations.
- B3: Start with dense heads and high-capacity layers, then tune dropout/weight decay incrementally.

Passing guideline:
- 80 percent or higher indicates strong readiness for enterprise project contribution.

## Tabular Models: Which Activation to Use and Why

### Activation selection guide for tabular data

1. Hidden layers in most tabular classifiers/regressors: `relu`
Purpose: fast, strong default, stable optimization in many enterprise datasets.
Use case: credit risk scoring, churn prediction, fraud triage.

2. Hidden layers when gradients are noisy or deep stack is unstable: `gelu` or `selu`
Purpose: smoother activation response can improve convergence behavior.
Use case: medium-to-large feature spaces with deeper MLP heads.

3. Hidden layers where negative inputs carry business signal: `leaky_relu`
Purpose: preserves a small gradient for negative region.
Use case: financial deltas where negative values remain informative.

4. Output layer for binary classification: `sigmoid`
Purpose: produces probability in [0, 1] for one target class.
Use case: default probability of customer churn.

5. Output layer for multi-class single-label classification: `softmax`
Purpose: normalized class probabilities across mutually exclusive classes.
Use case: support ticket routing to exactly one queue.

6. Output layer for regression: `linear` (or no activation)
Purpose: unrestricted numeric output.
Use case: revenue forecast, claim amount estimation.

Practical rule:
- Pick output activation by target type first.
- Then choose hidden activation (`relu` default, adjust to `gelu`/`leaky_relu` if training behavior suggests).

In [64]:
# Separate Example 1: Custom activation function for tabular fraud-risk scoring

import tensorflow as tf
import numpy as np

np.random.seed(55)
tf.random.set_seed(55)

# Use case: keep small negative signal instead of zeroing it out completely.
def scaled_leaky_relu(x, alpha=0.05, scale=1.1):
    return scale * tf.where(x >= 0.0, x, alpha * x)

# Synthetic tabular binary task
x_tab_fn = tf.random.normal([80, 10])
score = tf.reduce_sum(x_tab_fn[:, :3], axis=1) - 0.4 * tf.reduce_sum(x_tab_fn[:, 3:6], axis=1)
y_tab_fn = tf.cast(score > 0.0, tf.float32)

fn_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(10,)),
    tf.keras.layers.Dense(24, activation=scaled_leaky_relu),
    tf.keras.layers.Dense(12, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])

fn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
fn_hist = fn_model.fit(x_tab_fn, y_tab_fn, epochs=6, batch_size=16, verbose=0)

print('Custom function use case: fraud-risk style binary scoring')
print('Purpose: retain weak negative feature signals with controlled slope')
print('Final loss:', float(fn_hist.history['loss'][-1]))
print('Final acc:', float(fn_hist.history['accuracy'][-1]))

Custom function use case: fraud-risk style binary scoring
Purpose: retain weak negative feature signals with controlled slope
Final loss: 0.6572145223617554
Final acc: 0.6499999761581421


In [65]:
# Separate Example 2: Custom activation layer for tabular revenue-forecast robustness

import tensorflow as tf
import numpy as np

np.random.seed(56)
tf.random.set_seed(56)

# Use case: adaptively scale positive and negative regimes in volatile tabular signals.
class AdaptiveBiScaleActivation(tf.keras.layers.Layer):
    def __init__(self, init_pos=1.0, init_neg=0.2, **kwargs):
        super().__init__(**kwargs)
        self.init_pos = init_pos
        self.init_neg = init_neg

    def build(self, input_shape):
        self.pos_scale = self.add_weight(
            name='pos_scale',
            shape=(),
            initializer=tf.keras.initializers.Constant(self.init_pos),
            trainable=True,
        )
        self.neg_scale = self.add_weight(
            name='neg_scale',
            shape=(),
            initializer=tf.keras.initializers.Constant(self.init_neg),
            trainable=True,
        )

    def call(self, inputs):
        return tf.where(inputs >= 0.0, self.pos_scale * inputs, self.neg_scale * inputs)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'init_pos': self.init_pos, 'init_neg': self.init_neg})
        return cfg

# Synthetic tabular regression task
x_tab_layer = tf.random.normal([100, 12])
y_tab_layer = (
    1.6 * x_tab_layer[:, 0]
    - 0.9 * x_tab_layer[:, 1]
    + 0.5 * tf.square(x_tab_layer[:, 2])
    + 0.2 * tf.random.normal([100])
)

layer_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(12,)),
    tf.keras.layers.Dense(20),
    AdaptiveBiScaleActivation(),
    tf.keras.layers.Dense(10, activation='relu'),
    tf.keras.layers.Dense(1, activation='linear'),
])

layer_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
layer_hist = layer_model.fit(x_tab_layer, y_tab_layer, epochs=8, batch_size=20, verbose=0)

custom_act_layer = layer_model.layers[1]
print('Custom layer use case: revenue-forecast style regression under regime shifts')
print('Purpose: learn separate scaling for positive and negative activation regions')
print('Final MSE:', float(layer_hist.history['loss'][-1]))
print('Final MAE:', float(layer_hist.history['mae'][-1]))
print('Learned pos_scale:', float(custom_act_layer.pos_scale.numpy()))
print('Learned neg_scale:', float(custom_act_layer.neg_scale.numpy()))

Custom layer use case: revenue-forecast style regression under regime shifts
Purpose: learn separate scaling for positive and negative activation regions
Final MSE: 2.994586706161499
Final MAE: 1.3318989276885986
Learned pos_scale: 1.0202648639678955
Learned neg_scale: 0.20361372828483582
